# Lab Tuần 4 — Xử lý dữ liệu theo lô với PySpark


Yêu cầu trước khi bắt đầu: đã cài `pyspark` và JDK 11/17, và thư mục `data/`
nằm cùng cấp với thư mục chứa notebook này.

```bash
pip install pyspark
```
---

In [7]:
!unzip -o "/content/BaiTap_MapReduce_DaGiai_FULL.zip" -d "/content/"

Archive:  /content/BaiTap_MapReduce_DaGiai_FULL.zip
  inflating: /content/BaiTap_MapReduce_DaGiai/lab_tuan4_DA_GIAI.ipynb  
  inflating: /content/BaiTap_MapReduce_DaGiai/README.txt  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/access.log  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/transactions.log  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/sales.csv  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/orders.csv  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/document.txt  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/big_file.csv  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/customers.csv  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/events.jsonl  
  inflating: /content/BaiTap_MapReduce_DaGiai/data/temperatures.csv  


In [8]:
%cd /content/BaiTap_MapReduce_DaGiai

/content/BaiTap_MapReduce_DaGiai


In [9]:
!ls data

access.log    customers.csv  events.jsonl  sales.csv	     transactions.log
big_file.csv  document.txt   orders.csv    temperatures.csv


In [10]:
pip install pyspark

## Bài 0 — Khởi động

Mục tiêu: dựng `SparkSession`, kiểm tra môi trường và mở được Spark UI.

Chạy ô dưới. Nếu báo lỗi `JAVA_HOME`, cài JDK 11 hoặc 17 rồi chạy lại.
Ghi lại địa chỉ Spark UI — cả buổi lab sẽ dùng nó để quan sát Stage và Shuffle.

In [11]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import json, time, os

spark = (SparkSession.builder
         .appName("Lab-Tuan4")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

# Thu muc data nam cung cap voi notebook trong ZIP duoc cung cap.
DATA = "./data"
print("Spark", spark.version, "| so core:", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

Spark 4.0.4 | so core: 2
Spark UI: http://0977e19ef09a:4040


### Câu hỏi

Cụm đang chạy ở chế độ Cluster Manager nào? `local[*]` tương ứng bao nhiêu Executor?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

- Chương trình đang chạy ở **Spark local mode**, tức không dùng YARN/Kubernetes/Standalone cluster manager bên ngoài.
- `local[*]` yêu cầu Spark dùng **tất cả logical CPU cores khả dụng** trên máy làm worker threads. Số thread thực tế có thể xem qua `sc.defaultParallelism`.
- Ở local mode không có nhiều executor process như cụm thật; về thực tế là **một JVM/local executor cùng process với driver**, sử dụng nhiều worker threads.

## Bài 1 — WordCount và vai trò của Combiner

Dữ liệu: `document.txt` (300 dòng văn bản).

Viết WordCount bằng **hai cách** rồi so sánh chi phí:
- **1a** dùng `reduceByKey`
- **1b** dùng `groupByKey` + `mapValues`

Sau khi chạy, mở tab **Stages** trên Spark UI, tìm cột *Shuffle Read / Shuffle Write*
của hai job và ghi lại con số.

In [12]:
# Dung dung file document.txt cua de bai
doc_path = f"{DATA}/document.txt"
lines = sc.textFile(doc_path)

# Tach tu -> chuyen chu thuong -> loai token rong
words = (lines.flatMap(lambda line: line.lower().split())
              .filter(lambda w: w != ""))

# 1a: reduceByKey -- co map-side combine truoc khi shuffle
wc_reduce = (words.map(lambda w: (w, 1))
                  .reduceByKey(lambda a, b: a + b))

# 1b: groupByKey -- shuffle tat ca value theo key, sau do moi tinh tong
wc_group = (words.map(lambda w: (w, 1))
                 .groupByKey()
                 .mapValues(lambda values: sum(values)))

reduce_result = wc_reduce.collect()
group_result = wc_group.collect()

print("Top 10:", sorted(reduce_result, key=lambda x: -x[1])[:10])
print("Hai cach cho cung ket qua:",
      sorted(reduce_result) == sorted(group_result))

# Sau khi chay cell nay, mo Spark UI -> Stages va ghi Shuffle Read/Write.
# reduceByKey thuong shuffle it du lieu hon do co map-side combine;
# groupByKey phai dua tat ca value cua cung key qua mang truoc khi cong.


Top 10: [('du', 317), ('lieu', 317), ('nao', 148), ('den', 147), ('mapreduce', 124), ('nho', 118), ('vi', 118), ('thay', 118), ('giai', 113), ('doan', 113)]
Hai cach cho cung ket qua: True


### Câu hỏi

Hai cách cho cùng kết quả. Vậy khác nhau ở đâu, và vì sao `reduceByKey` lại rẻ hơn?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**


Hai cách cho cùng kết quả logic nhưng chi phí shuffle khác nhau.

Trong lần chạy thực tế:
- `reduceByKey`: Shuffle Read = **1842 bytes**, Shuffle Write = **1842 bytes**.
- `groupByKey`: Shuffle Read = **2371 bytes**, Shuffle Write = **2371 bytes**.

Như vậy `groupByKey` truyền nhiều hơn 529 bytes, tương đương khoảng **28.7%**.

Nguyên nhân là `groupByKey()` phải chuyển toàn bộ các value của mỗi key qua mạng rồi mới cộng ở reducer. Trong khi đó, `reduceByKey()` thực hiện map-side combine, cộng các giá trị cùng key ngay tại mỗi partition trước khi shuffle. Vì vậy số record truyền qua mạng giảm, tiết kiệm network, memory và thời gian xử lý.

## Bài 2 — phép trung bình

Dữ liệu: `temperatures.csv` — cột `station_id, year, temperature`.

Số bản ghi giữa các trạm **lệch rất mạnh**: TP.HCM có hơn 4000 bản ghi,
đỉnh Fansipan chỉ có 3. Đây là bài quan trọng nhất của buổi lab.

Tính nhiệt độ trung bình toàn quốc bằng **hai cách** rồi so sánh:
- **2a — cách sai**: tính trung bình của từng trạm trước, rồi lấy trung bình của các trung bình đó
- **2b — cách đúng**: dùng `aggregateByKey` giữ cặp `(tổng, số lượng)`, chỉ chia ở bước cuối

In [13]:
raw = sc.textFile(f"{DATA}/temperatures.csv")
header = raw.first()
pairs = (raw.filter(lambda l: l != header)
            .map(lambda l: l.split(","))
            .map(lambda c: (c[0], float(c[2]))))      # (station_id, temperature)
print("So ban ghi:", pairs.count())

# Ham cong don (tong, so_luong)
seq_op = lambda acc, t: (acc[0] + t, acc[1] + 1)
comb_op = lambda a, b: (a[0] + b[0], a[1] + b[1])

# 2a: CACH SAI -- moi tram co trong so bang nhau bat chap so ban ghi
thong_ke_tram = pairs.aggregateByKey((0.0, 0), seq_op, comb_op)
tb_tung_tram = thong_ke_tram.mapValues(lambda x: x[0] / x[1])
tb_sai = tb_tung_tram.values().mean()

# 2b: CACH DUNG -- giu (tong, so_luong), gop tat ca roi moi chia mot lan o cuoi
# aggregateByKey van duoc dung de ket hop an toan theo partition/key.
tong_toan_quoc, so_luong_toan_quoc = (thong_ke_tram.values()
                                       .reduce(lambda a, b: (a[0] + b[0], a[1] + b[1])))
tb_dung = tong_toan_quoc / so_luong_toan_quoc

print("Trung binh SAI :", round(tb_sai, 2))
print("Trung binh DUNG:", round(tb_dung, 2))
print("Chenh lech     :", round(abs(tb_dung - tb_sai), 2), "do C")

So ban ghi: 6001
Trung binh SAI : 22.16
Trung binh DUNG: 28.08
Chenh lech     : 5.92 do C


> **Kết quả mong đợi:** Trung binh SAI ≈ 22.16 | DUNG ≈ 28.08 | chênh ≈ 5.92 °C

### Câu hỏi

Vì sao phép trung bình không dùng trực tiếp làm Combiner được, trong khi SUM, MAX, COUNT thì được?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

`SUM`, `MAX`, `COUNT` có thể combine trực tiếp vì chúng có tính chất kết hợp: có thể tính từng phần rồi gộp các kết quả trung gian mà vẫn ra đúng kết quả cuối.

Riêng **trung bình cộng không thể chỉ giữ một số `avg` rồi lấy trung bình tiếp**, vì các nhóm có thể có số phần tử khác nhau. Ví dụ trung bình của 4000 mẫu và trung bình của 3 mẫu không được có cùng trọng số.

Muốn dùng Combiner đúng cho average, phải biến trạng thái trung gian thành **`(sum, count)`**. Hai trạng thái này gộp được theo `(sum1 + sum2, count1 + count2)`, sau cùng mới tính `sum / count`.

Với dữ liệu đề bài: cách sai ≈ **22.16°C**, cách đúng ≈ **28.08°C**, chênh ≈ **5.92°C**.

## Bài 3 — Accumulator và dữ liệu bẩn

Dữ liệu: `events.jsonl` — JSON Lines, trong đó có một số dòng **cố tình hỏng**.

Dùng `Accumulator` đếm số bản ghi không parse được **ngay trong lúc xử lý**,
không dùng thêm một Action riêng để đếm.

In [14]:
loi_parse = sc.accumulator(0)

def parse_an_toan(dong):
    try:
        return json.loads(dong)
    except (json.JSONDecodeError, TypeError, ValueError):
        loi_parse.add(1)
        return None

ban_ghi = (sc.textFile(f"{DATA}/events.jsonl")
             .map(parse_an_toan)
             .filter(lambda x: x is not None))

# Action count() kich hoat lineage, luc nay accumulator moi duoc cap nhat.
print("So ban ghi hop le:", ban_ghi.count())
print("So ban ghi loi   :", loi_parse.value)

So ban ghi hop le: 2844
So ban ghi loi   : 156


> **Kết quả mong đợi:** 2844 bản ghi hợp lệ, 156 bản ghi lỗi

### Câu hỏi

Nếu đọc `loi_parse.value` **trước** khi gọi `count()` thì được bao nhiêu? Và vì sao con số này có thể sai khi chạy trên cụm thật?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

Nếu đọc `loi_parse.value` **trước** `count()` thì giá trị là **0**, vì Spark dùng lazy evaluation: `map()` và `filter()` mới chỉ tạo lineage, chưa thực sự đọc dữ liệu.

Sau `count()`, với bộ dữ liệu này ta nhận **2844 bản ghi hợp lệ** và **156 bản ghi lỗi**.

Trên cụm thật, accumulator không nên được xem như nguồn dữ liệu nghiệp vụ chính xác tuyệt đối khi nó được cập nhật bên trong transformation. Task/stage có thể bị chạy lại do lỗi, recomputation hoặc nhiều action, làm cùng record có khả năng cập nhật accumulator nhiều lần. Accumulator phù hợp nhất cho telemetry/debug/counter; nếu cần kết quả chính xác để xử lý tiếp thì nên biểu diễn lỗi thành RDD/DataFrame rồi aggregate bằng phép biến đổi Spark thông thường.

## Bài 4 — DataFrame, Join và left_anti

Dữ liệu: `customers.csv` (600 khách) và `orders.csv`.

Tìm những khách hàng **chưa từng đặt đơn nào**, bằng hai cách,
rồi so sánh kế hoạch thực thi của chúng bằng `.explain()`.

In [15]:
kh = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
dh = spark.read.csv(f"{DATA}/orders.csv",    header=True, inferSchema=True)

# 4a: left_anti -- chi lay row ben trai khong co key khop o ben phai
khach_da_mua = dh.select("customer_id").distinct()
chua_mua_a = kh.join(khach_da_mua, on="customer_id", how="left_anti")

# 4b: left join roi loc cac row khong match (order_id se null)
chua_mua_b = (kh.join(dh, on="customer_id", how="left")
                .filter(F.col("order_id").isNull())
                .select(kh["customer_id"], kh["name"], kh["city"], kh["segment"]))

print("left_anti       :", chua_mua_a.count())
print("left join + null:", chua_mua_b.count())

print("\n=== EXPLAIN left_anti ===")
chua_mua_a.explain(mode="formatted")
print("\n=== EXPLAIN left join + null ===")
chua_mua_b.explain(mode="formatted")

left_anti       : 151
left join + null: 151

=== EXPLAIN left_anti ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- BroadcastHashJoin LeftAnti BuildRight (8)
   :- Scan csv  (1)
   +- BroadcastExchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Filter (3)
                  +- Scan csv  (2)


(1) Scan csv 
Output [4]: [customer_id#17, name#18, city#19, segment#20]
Batched: false
Location: InMemoryFileIndex [file:/content/BaiTap_MapReduce_DaGiai/data/customers.csv]
ReadSchema: struct<customer_id:int,name:string,city:string,segment:string>

(2) Scan csv 
Output [1]: [customer_id#39]
Batched: false
Location: InMemoryFileIndex [file:/content/BaiTap_MapReduce_DaGiai/data/orders.csv]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int>

(3) Filter
Input [1]: [customer_id#39]
Condition : isnotnull(customer_id#39)

(4) HashAggregate
Input [1]: [customer_id#39]
Keys [1]: [customer_id#39]
Functions: []
Aggregat

> **Kết quả mong đợi:** Cả hai cách đều ra 151 khách hàng

### Câu hỏi

Hai cách cho cùng kết quả. Cách nào rẻ hơn và vì sao?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

`left_anti` là phép toán đúng trực tiếp với câu hỏi “row bên trái nào **không có match** bên phải”, nên thường là cách rõ ràng và gọn hơn. Engine có thể chỉ giữ thông tin cần thiết để kiểm tra sự tồn tại của key bên phải thay vì tạo toàn bộ kết quả của một `left outer join` rồi mới lọc `NULL`.

Cách `left join + isNull()` về logic cho cùng kết quả nhưng biểu thức ban đầu tạo nhiều cột/row trung gian hơn. Tuy nhiên Spark Catalyst có thể tối ưu mạnh và ở một số phiên bản hai cách có thể cho physical plan rất giống nhau. Vì vậy nên kiểm tra `.explain()` thay vì mặc định rằng chênh lệch luôn lớn.

Với dữ liệu đề bài, cả hai cách đều cho **151 khách hàng chưa từng đặt đơn**.

## Bài 5 — Data skew và Custom Partitioner

Dữ liệu: `access.log` (định dạng `level ip method endpoint status latency`)
và `big_file.csv`.

**5a** Đếm số lượt truy cập theo `endpoint` và chỉ ra hiện tượng lệch tải.
**5b** Phân vùng `big_file.csv` theo domain của email bằng Custom Partitioner,
rồi đo **kích thước từng partition** để thấy phân bố không đều.

In [16]:
# 5a -- do lech tai theo endpoint
logs = sc.textFile(f"{DATA}/access.log")
phan_bo = (logs.map(lambda line: line.split())
               .filter(lambda c: len(c) >= 4)
               .map(lambda c: (c[3], 1))
               .reduceByKey(lambda a, b: a + b)
               .sortBy(lambda x: -x[1])
               .collect())

for ep, n in phan_bo:
    print("%-16s %5d" % (ep, n))

# 5b -- custom partitioner theo domain email
# Anh xa 5 domain co trong du lieu sang 5 partition de ket qua on dinh,
# tranh phu thuoc randomized hash cua Python.
DOMAIN_TO_PARTITION = {
    "gmail.com": 0,
    "outlook.com": 1,
    "yahoo.jp": 2,
    "vnu.edu.vn": 3,
    "hcmute.edu.vn": 4,
}

def theo_domain(key):
    domain = key.rsplit("@", 1)[-1].lower()
    return DOMAIN_TO_PARTITION.get(domain, sum(ord(ch) for ch in domain) % 5)

df = spark.read.csv(f"{DATA}/big_file.csv", header=True, inferSchema=True)
rdd = df.rdd.map(lambda r: (r["email"], r["amount"]))

phan_vung = rdd.partitionBy(5, theo_domain)
kich_thuoc = phan_vung.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
print("Kich thuoc tung partition:", kich_thuoc)

/api/search       3296
/api/product      1050
/api/cart          709
/api/login         487
/api/checkout      328
/api/admin         130
Kich thuoc tung partition: [20048, 20105, 20016, 19661, 20170]


> **Kết quả mong đợi:** `/api/search` chiếm ~55%, gấp ~25 lần endpoint nhỏ nhất

### Câu hỏi

Trong 5a, nếu mỗi endpoint đi về một Reducer riêng thì điều gì xảy ra với tổng thời gian chạy?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

Nếu mỗi endpoint bị đưa nguyên về một reducer riêng thì reducer xử lý `/api/search` sẽ nhận khoảng **3296/6000 = 54.9%** toàn bộ request, trong khi `/api/admin` chỉ có **130** request. Reducer của `/api/search` trở thành **straggler**: các reducer nhỏ đã xong nhưng stage vẫn phải chờ reducer lớn nhất hoàn thành, nên tổng thời gian gần như bị chi phối bởi partition nặng nhất.

Đây là hiện tượng **data skew**. Cách xử lý tùy bài toán có thể gồm tăng số partition, “salting” hot key để chia một key nóng thành nhiều key tạm rồi gộp lại, hoặc thiết kế partitioner phù hợp hơn.

## Bài 6 (nếu còn thời gian) — Đo tác động của cache()

Đo thời gian chạy `count()` hai lần trên cùng một RDD, khi **chưa** cache và khi **đã** cache.

In [17]:
big = sc.textFile(f"{DATA}/big_file.csv").filter(lambda l: "gmail.com" in l)

# CHUA cache: moi action phai doc/loc lai lineage
start = time.perf_counter()
n1 = big.count()
t1 = time.perf_counter() - start

start = time.perf_counter()
n2 = big.count()
t2 = time.perf_counter() - start

print(f"Khong cache - count 1: {n1} rows, {t1:.4f} s")
print(f"Khong cache - count 2: {n2} rows, {t2:.4f} s")

# CO cache: action dau de materialize cache; action sau moi la phep do doc tu cache
big_cached = big.cache()
start = time.perf_counter()
n_load = big_cached.count()
t_load = time.perf_counter() - start

start = time.perf_counter()
n_cached = big_cached.count()
t_cached = time.perf_counter() - start

print(f"Nap cache            : {n_load} rows, {t_load:.4f} s")
print(f"Da cache - count lai : {n_cached} rows, {t_cached:.4f} s")

big_cached.unpersist()

Khong cache - count 1: 20048 rows, 0.4437 s
Khong cache - count 2: 20048 rows, 0.2668 s
Nap cache            : 20048 rows, 0.3760 s
Da cache - count lai : 20048 rows, 0.0990 s


PythonRDD[107] at RDD at PythonRDD.scala:57

### Câu hỏi

Khi chưa cache, vì sao lần `count()` thứ hai vẫn tốn gần bằng lần đầu?

*(Trả lời vào ô markdown bên dưới)*

**Trả lời:**

Khi chưa cache, RDD chỉ lưu **lineage** (các bước cần thực hiện), không lưu sẵn dữ liệu sau transformation. Vì `count()` là một action, mỗi lần gọi Spark phải đọc lại `big_file.csv`, chạy lại `filter()` và đếm lại từ đầu. Do đó lần `count()` thứ hai thường vẫn tốn gần bằng lần đầu (chênh lệch nhỏ có thể đến từ OS file cache/JVM warm-up).

Sau `.cache()`, lần action đầu tiên dùng để **materialize** dữ liệu vào cache. Từ action tiếp theo Spark có thể đọc partition đã cache thay vì recompute toàn bộ lineage, nên thường nhanh hơn rõ rệt.

---
## Kết thúc / ghi chú nộp bài

- Các kết quả dữ liệu đã kiểm tra từ bộ ZIP: Bài 2 = **22.16 / 28.08 / 5.92°C**; Bài 3 = **2844 hợp lệ / 156 lỗi**; Bài 4 = **151**; Bài 5a `/api/search` = **3296/6000 (~54.9%)**.
- Bài 1 hiện đã dùng đúng `data/document.txt` do bạn bổ sung; không còn fallback sang `access.log`.
- Shuffle Read/Write và thời gian cache phải lấy từ lần chạy Spark thực tế trên máy của bạn; đây là số đo môi trường, không phải đáp án cố định.

```python
spark.stop()
```